In [1]:
import pandas as pd
import numpy as np


In [2]:
master_table = pd.read_csv("data/aadhaar_master_table.csv", parse_dates=["date"])

value_cols = [
    "age_0_5", "age_5_17", "age_18_greater",
    "demo_age_5_17", "demo_age_17_",
    "bio_age_5_17", "bio_age_17_"
]

In [3]:
master_table = master_table[
    ~((master_table["state"] == "100000") &
      (master_table["district"] == "100000"))
]

In [4]:
len(master_table)

334764

In [5]:
monthly_base = master_table.copy()

monthly_base["month"] = monthly_base["date"].dt.to_period("M").dt.to_timestamp()

monthly_base = (
    monthly_base
    .groupby(["state", "district", "month"], as_index=False)
    .agg({
        "age_0_5": "sum",
        "age_5_17": "sum",
        "age_18_greater": "sum",
        "demo_age_5_17": "sum",
        "demo_age_17_": "sum",
        "bio_age_5_17": "sum",
        "bio_age_17_": "sum"
    })
)


In [6]:
monthly_base["new_enrolments"] = (
    monthly_base["age_0_5"] +
    monthly_base["age_5_17"] +
    monthly_base["age_18_greater"]
)

monthly_base["total_updates"] = (
    monthly_base["demo_age_5_17"] +
    monthly_base["demo_age_17_"] +
    monthly_base["bio_age_5_17"] +
    monthly_base["bio_age_17_"]
)

monthly_base["total_activity"] = (
    monthly_base["new_enrolments"] +
    monthly_base["total_updates"]
)


Stress = “A ratio capturing the balance between corrective workload (updates) and expansion workload (new enrolments) at the district level.”

In [7]:
### Calculate Aadhar Infrastructure stress Index
monthly_base["stress_index"] = (
    monthly_base["total_updates"] /
    (monthly_base["new_enrolments"] + 1)
)


“We compute infrastructure stress monthly to capture operational load dynamics and then aggregate it to characterize each district’s typical and worst-case system pressure.”

In [8]:
district_stress_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_stress_index=("stress_index", "mean"),
        max_stress_index=("stress_index", "max"),
        stress_volatility=("stress_index", "std"),
        active_months=("stress_index", "count")
    )
)



In [9]:
district_stress_summary.head(10)

,state,district,mean_stress_index,max_stress_index,stress_volatility,active_months
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,259.000000,97.416098,10
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,4.000000,1.257201,10
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,21.666667,9.163387,10
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,671.000000,207.028878,10
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,1437.000000,563.351927,10
5,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,674.567832,1922.000000,725.666597,10
6,ANDHRA PRADESH,ADILABAD,2952.720898,9603.000000,3523.162448,10
7,ANDHRA PRADESH,ALLURI SITHARAMA RAJU,1600.868172,6948.000000,2209.379973,10
8,ANDHRA PRADESH,ANAKAPALLI,835.340743,3616.000000,1143.694626,10
9,ANDHRA PRADESH,ANANTAPUR,7389.462129,29828.000000,9676.180510,10


In [10]:
district_stress_summary.to_csv("data/district_summary/district_stress_summary.csv", index=False)

### Updates dependency ratio

The Update Dependency Ratio (UDR) answers:

“What share of Aadhaar activity in a district is spent correcting existing records rather than onboarding new residents

In [11]:
# Calculating Updates dependency ratio
monthly_base["update_dependency_ratio"] = (
    monthly_base["total_updates"] /
    (monthly_base["total_activity"] + 1)
)


In [12]:
monthly_base[[
    "state", "district", "month",
    "total_updates", "total_activity",
    "update_dependency_ratio"
]].head()


,state,district,month,total_updates,total_activity,update_dependency_ratio
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,209.0,209.0,0.995238
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,184.0,184.0,0.994595
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,180.0,180.0,0.994475
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,140.0,140.0,0.992908
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,259.0,259.0,0.996154


In [13]:
# Aggregate Update Dependency Ratio → district level
district_dependency_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_update_dependency=("update_dependency_ratio", "mean"),
        max_update_dependency=("update_dependency_ratio", "max"),
        dependency_volatility=("update_dependency_ratio", "std")
    )
)


In [14]:
district_dependency_summary.head()

,state,district,mean_update_dependency,max_update_dependency,dependency_volatility
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.879568,0.996154,0.309773
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.163333,0.800000,0.285644
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.376071,0.955882,0.485612
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.873744,0.998512,0.308890
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.884397,0.999305,0.311274


In [15]:
district_dependency_summary.to_csv("data/district_summary/district_dependency_summary.csv", index=False)

### Dormancy Index

The Dormancy Index answers:

“How consistently is Aadhaar infrastructure actually used in a district over time?”

```Dormancy Index	Interpretation
~0.0	Infrastructure consistently active
0.2 – 0.4	Intermittent usage
> 0.5	Largely dormant
~1.0	Infrastructure present but unused```

In [16]:
monthly_base["is_active"] = (monthly_base["total_activity"] > 0).astype(int)

In [17]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,stress_index,update_dependency_ratio,is_active
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1


In [18]:
district_dormancy_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        active_months=("is_active", "sum"),
        total_months=("is_active", "count")
    )
)

district_dormancy_summary["pct_active_months"] = (
    district_dormancy_summary["active_months"] /
    district_dormancy_summary["total_months"]
)

district_dormancy_summary["dormancy_index"] = (
    1 - district_dormancy_summary["pct_active_months"]
)


In [19]:
district_dormancy_summary.head()

,state,district,active_months,total_months,pct_active_months,dormancy_index
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,9,10,0.9,0.1
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,3,10,0.3,0.7
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,4,10,0.4,0.6
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,9,10,0.9,0.1
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,9,10,0.9,0.1


In [20]:
def longest_dormant_streak(x):
    max_streak = streak = 0
    for v in x:
        if v == 0:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

dormant_streaks = (
    monthly_base
    .sort_values("month")
    .groupby(["state", "district"])["is_active"]
    .apply(longest_dormant_streak)
    .reset_index(name="max_dormant_streak")
)

district_dormancy_summary = district_dormancy_summary.merge(
    dormant_streaks,
    on=["state", "district"],
    how="left"
)


In [21]:
district_dormancy_summary

,state,district,active_months,total_months,pct_active_months,dormancy_index,max_dormant_streak
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,9,10,0.9,0.1,1
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,3,10,0.3,0.7,6
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,4,10,0.4,0.6,6
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,9,10,0.9,0.1,1
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,9,10,0.9,0.1,1
...,...,...,...,...,...,...,...
1089,WEST BENGAL,WEST MEDINIPUR,4,10,0.4,0.6,6
1090,WEST BENGAL,WEST MIDNAPORE,9,10,0.9,0.1,1
1091,WEST BENGLI,HOOGHLY,2,10,0.2,0.8,8
1092,WESTBENGAL,HOOGHLY,4,10,0.4,0.6,6


In [22]:
district_dormancy_summary.to_csv("data/district_summary/district_dormancy_summary.csv")

### BDS (Biometric Decay Score)



In [23]:
monthly_base["adult_total_updates"] = (
    monthly_base["bio_age_17_"] +
    monthly_base["demo_age_17_"]
)

monthly_base["bds"] = (
    monthly_base["bio_age_17_"] /
    (monthly_base["adult_total_updates"] + 1)
)


In [24]:
district_bds_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_bds=("bds", "mean"),
        max_bds=("bds", "max")
    )
)

In [25]:
district_bds_summary.head()

,state,district,mean_bds,max_bds
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.711003,0.995833
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.050000,0.500000
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.205137,0.580645
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.603048,0.993243
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.547936,0.997797


In [26]:
district_bds_summary.to_csv("data/district_summary/district_bds_summary.csv", index=False)

### Child Compliance Index (CCI)

In [27]:
monthly_base["child_total_activity"] = (
    monthly_base["age_5_17"] +
    monthly_base["demo_age_5_17"] +
    monthly_base["bio_age_5_17"]
)

monthly_base["cci"] = (
    monthly_base["bio_age_5_17"] /
    (monthly_base["child_total_activity"] + 1)
)


In [28]:
district_cci_summary = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_cci=("cci", "mean"),
        min_cci=("cci", "min")
    )
)


In [29]:
district_cci_summary.head(10)

,state,district,mean_cci,min_cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,0.848777,0.0
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.050000,0.0
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,0.342552,0.0
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,0.831662,0.0
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,0.871154,0.0
5,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,0.845995,0.0
6,ANDHRA PRADESH,ADILABAD,0.754645,0.0
7,ANDHRA PRADESH,ALLURI SITHARAMA RAJU,0.750180,0.0
8,ANDHRA PRADESH,ANAKAPALLI,0.597806,0.0
9,ANDHRA PRADESH,ANANTAPUR,0.731795,0.0


In [30]:
district_cci_summary.to_csv("data/district_summary/district_cci_summary.csv", index=False)

In [ ]:
monthly_base.to_csv("data/monthly_metrics.csv", index=False)

### Building the Digital Maturity feature table

In [32]:
digital_maturity_df = (
    district_stress_summary
    .merge(district_dependency_summary, on=["state", "district"], how="left")
    .merge(district_dormancy_summary, on=["state", "district"], how="left")
)


In [33]:
from scipy.stats import linregress

def compute_growth(group):
    x = np.arange(len(group))
    y = group["total_activity"].values
    if len(y) < 2:
        return 0.0
    return linregress(x, y).slope

growth_df = (
    monthly_base
    .sort_values("month")
    .groupby(["state", "district"])
    .apply(compute_growth)
    .reset_index(name="activity_growth_slope")
)

digital_maturity_df = digital_maturity_df.merge(
    growth_df, on=["state", "district"], how="left"
)


/tmp/ipykernel_123353/4017408124.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_growth)


In [34]:
digital_maturity_df

,state,district,mean_stress_index,max_stress_index,stress_volatility,active_months_x,mean_update_dependency,max_update_dependency,dependency_volatility,active_months_y,total_months,pct_active_months,dormancy_index,max_dormant_streak,activity_growth_slope
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,259.000000,97.416098,10,0.879568,0.996154,0.309773,9,10,0.9,0.1,1,42.703030
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,4.000000,1.257201,10,0.163333,0.800000,0.285644,3,10,0.3,0.7,6,0.260606
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,21.666667,9.163387,10,0.376071,0.955882,0.485612,4,10,0.4,0.6,6,26.375758
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,671.000000,207.028878,10,0.873744,0.998512,0.308890,9,10,0.9,0.1,1,-9.145455
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,1437.000000,563.351927,10,0.884397,0.999305,0.311274,9,10,0.9,0.1,1,-12.933333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089,WEST BENGAL,WEST MEDINIPUR,1.466667,7.000000,2.515139,10,0.277500,0.875000,0.379247,4,10,0.4,0.6,6,0.606061
1090,WEST BENGAL,WEST MIDNAPORE,976.959575,2762.000000,1074.952371,10,0.878452,0.999638,0.310034,9,10,0.9,0.1,1,1274.193939
1091,WEST BENGLI,HOOGHLY,0.300000,2.000000,0.674949,10,0.116667,0.666667,0.249072,2,10,0.2,0.8,8,0.151515
1092,WESTBENGAL,HOOGHLY,6.866667,34.000000,11.410630,10,0.368235,0.971429,0.476377,4,10,0.4,0.6,6,7.400000


In [35]:
digital_maturity_features = digital_maturity_df[
    [
        "state",
        "district",
        "mean_stress_index",
        "mean_update_dependency",
        "stress_volatility",
        "dependency_volatility",
        "pct_active_months",
        "activity_growth_slope"
    ]
]


In [36]:
digital_maturity_features.head()

,state,district,mean_stress_index,mean_update_dependency,stress_volatility,dependency_volatility,pct_active_months,activity_growth_slope
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,106.583631,0.879568,97.416098,0.309773,0.9,42.703030
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.550000,0.163333,1.257201,0.285644,0.3,0.260606
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,6.728472,0.376071,9.163387,0.485612,0.4,26.375758
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,149.532449,0.873744,207.028878,0.308890,0.9,-9.145455
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,501.677162,0.884397,563.351927,0.311274,0.9,-12.933333


In [37]:
digital_maturity_features.isna().sum()


state                     0
district                  0
mean_stress_index         0
mean_update_dependency    0
stress_volatility         0
dependency_volatility     0
pct_active_months         0
activity_growth_slope     0
dtype: int64

In [38]:
digital_maturity_features.to_csv("data/digital_maturity_features.csv", index=False)

In [39]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,stress_index,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381


### Mean_monthly activity

In [ ]:
activity_heatmap_df = (
    monthly_base
    .groupby(["state", "district"], as_index=False)
    .agg(
        mean_monthly_activity=("total_activity", "mean")
    )
)


In [41]:
activity_heatmap_df.head()

,state,district,mean_monthly_activity
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,284.0
1,ANDAMAN & NICOBAR ISLANDS,NICOBARS,0.7
2,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,71.0
3,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,267.3
4,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,852.4


In [47]:
activity_heatmap_df.to_csv("data/mean_monthly_activity.csv", index=False)

In [43]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,new_enrolments,total_updates,total_activity,stress_index,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,0.0,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,0.0,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,0.0,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,0.0,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,0.0,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381


### Capped the stess index for visualization

In [44]:
cap = monthly_base["stress_index"].quantile(0.99)
monthly_base["stress_index_capped"] = monthly_base["stress_index"].clip(upper=cap)


In [45]:
monthly_base.head()

,state,district,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,...,total_updates,total_activity,stress_index,update_dependency_ratio,is_active,adult_total_updates,bds,child_total_activity,cci,stress_index_capped
0,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,...,209.0,209.0,209.0,0.995238,1,193.0,0.994845,16.0,0.941176,209.0
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-04-01,0.0,0.0,0.0,0.0,0.0,17.0,167.0,...,184.0,184.0,184.0,0.994595,1,167.0,0.994048,17.0,0.944444,184.0
2,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-05-01,0.0,0.0,0.0,0.0,0.0,22.0,158.0,...,180.0,180.0,180.0,0.994475,1,158.0,0.993711,22.0,0.956522,180.0
3,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-06-01,0.0,0.0,0.0,0.0,0.0,11.0,129.0,...,140.0,140.0,140.0,0.992908,1,129.0,0.992308,11.0,0.916667,140.0
4,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-07-01,0.0,0.0,0.0,0.0,0.0,20.0,239.0,...,259.0,259.0,259.0,0.996154,1,239.0,0.995833,20.0,0.952381,259.0


In [48]:
monthly_base.to_csv("data/monthly_metrics.csv", index=False)